The purpose of this notebook is to demonstrate consolidating the functions seen in the previous notebooks into one function called ```classify_sentiment(company_in)``` that takes the name of the company to be analysed (in our case 'uber' or 'blackstones') and carries out an end-to-end pipeline to classify the sentiment.

In theory this function could be further refined to take in another option ```LLM_in``` that specifies which LLM model backend to use (in our case DeepSeek or ChatGPT).



In [ ]:
def build_few_shot_prompt(text, example_dict):
    label_description = {
        -2: "Strongly Negative",
        -1: "Negative",
        0: "Neutral",
        1: "Positive",
        2: "Strongly Positive"
    }

    example_block = "Here are example paragraphs labeled by a human annotator:\n\n"
    for label, texts in example_dict.items():
        for t in texts:
            example_block += f"Sentence: {t.strip()}\nLabel: {label} ({label_description[label]})\n\n"

    classification_prompt = (
        f"{example_block}"
        "Now classify the sentiment of the following paragraph using one of these exact labels: -2, -1, 0, +1, +2.\n"
        "-2 = Strongly Negative\n"
        "-1 = Negative\n"
        " 0 = Neutral\n"
        "+1 = Positive\n"
        "+2 = Strongly Positive\n\n"
        "**Important Rules:**\n"
        "- Only return the numeric label: -2, -1, 0, +1, or +2.\n"
        "- Do not explain.\n\n"
        f"Paragraph: {text.strip()}\nLabel:"
    )

    return classification_prompt


In [ ]:
def classify_ordinal_sentiment(row, example_dict):
    prompt = build_few_shot_prompt(row["Paragraph Text"], example_dict)

    try:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "You are a financial sentiment classifier. Use only human-labeled examples to make consistent predictions."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=5,
            temperature=0
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: {e}"

In [ ]:
def classify_sentiment(company_in):
  if company_in == 'uber':
    dpath = uber_path
  elif company_in == 'bs':
    dpath = bs_path

  df_Empty = pd.read_excel(dpath, engine='openpyxl')
  client = OpenAI(api_key='', project='' )  # API key deleted here.

  start_time = time.time()  # Start the timer
  df_Empty["LLM Sentiment"] = df_Empty.apply(lambda row: classify_ordinal_sentiment(row, example_dict), axis=1)
  end_time = time.time()  # End the timer
  elapsed_time = end_time - start_time
  print(f"Time taken: {elapsed_time:.2f} seconds")
  return df_Empty